# Libraries

In [1]:
import os
import yaml
import json
import cv2

print("Libraries imported successfully: os, yaml, json, cv2")

def load_class_names(yaml_path):
    """Loads class names from a data.yaml file."""
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    return data['names']

print("Helper function 'load_class_names' defined.")

def yolo_to_coco(x_center, y_center, width, height, img_width, img_height):
    """Converts YOLO normalized bounding box coordinates to COCO absolute coordinates.
    Args:
        x_center, y_center, width, height: Normalized YOLO coordinates (float between 0 and 1).
        img_width, img_height: Original image dimensions.
    Returns:
        Tuple (x_min, y_min, coco_width, coco_height) in absolute pixel values.
    """
    # Convert normalized to absolute coordinates
    abs_x_center = x_center * img_width
    abs_y_center = y_center * img_height
    abs_width = width * img_width
    abs_height = height * img_height

    # Calculate COCO format (x_min, y_min, width, height)
    x_min = abs_x_center - (abs_width / 2)
    y_min = abs_y_center - (abs_height / 2)

    return int(x_min), int(y_min), int(abs_width), int(abs_height)

print("Helper function 'yolo_to_coco' defined.")

Libraries imported successfully: os, yaml, json, cv2
Helper function 'load_class_names' defined.
Helper function 'yolo_to_coco' defined.


# Subset Components Only

In [2]:
all_folds_data = {}

for i in range(5):
    fold_name = f'fold_{i}'
    fold_dir = os.path.join('/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data', fold_name)
    print(f"\nProcessing {fold_name}...")

    # Ensure fold_dir exists
    os.makedirs(fold_dir, exist_ok=True)

    # Define base paths for images and labels
    train_image_dir = os.path.join(fold_dir, 'train', 'images')
    train_label_dir = os.path.join(fold_dir, 'train', 'labels')
    valid_image_dir = os.path.join(fold_dir, 'valid', 'images')
    valid_label_dir = os.path.join(fold_dir, 'valid', 'labels')

    # Construct path to data.yaml
    data_yaml_path = os.path.join(fold_dir, 'data.yaml')

    # Create a dummy data.yaml if it doesn't exist to prevent FileNotFoundError
    if not os.path.exists(data_yaml_path):
        dummy_data = {
            'names': ['class1', 'class2', 'class3'],
            'nc': 3
        }
        with open(data_yaml_path, 'w') as f:
            yaml.safe_dump(dummy_data, f)
        print(f"Created dummy data.yaml at {data_yaml_path}.")

    # Load class names
    class_names = load_class_names(data_yaml_path)
    print(f"Loaded {len(class_names)} class names from {data_yaml_path}.")

    # Create COCO categories list
    categories = []
    for idx, name in enumerate(class_names):
        categories.append({
            'id': idx,
            'name': name,
            'supercategory': 'none'
        })
    print(f"Generated {len(categories)} COCO categories.")

    # Store paths and categories for the current fold
    all_folds_data[fold_name] = {
        'train_image_dir': train_image_dir,
        'train_label_dir': train_label_dir,
        'valid_image_dir': valid_image_dir,
        'valid_label_dir': valid_label_dir,
        'categories': categories,
        'class_names': class_names # Storing for potential debugging/future use
    }


Processing fold_0...
Loaded 23 class names from /content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_0/data.yaml.
Generated 23 COCO categories.

Processing fold_1...
Loaded 23 class names from /content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_1/data.yaml.
Generated 23 COCO categories.

Processing fold_2...
Loaded 23 class names from /content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_2/data.yaml.
Generated 23 COCO categories.

Processing fold_3...
Loaded 23 class names from /content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_3/data.yaml.
Generated 23 COCO categories.

Processing fold_4...
Loaded 23 class names from /content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_4/data.yaml.
Generated 23 COCO categories.


** Convert Train Data to COCO**




In [ ]:
for fold_name, fold_data in all_folds_data.items():
    print(f"\nConverting train data for {fold_name} to COCO format...")

    train_image_dir = fold_data['train_image_dir']
    train_label_dir = fold_data['train_label_dir']
    categories = fold_data['categories']
    fold_dir = os.path.dirname(train_image_dir.replace('/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_0/train/images', '/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_0/train/labels'))

    coco_output = {
        'images': [],
        'annotations': [],
        'categories': categories
    }

    image_id_counter = 0
    annotation_id_counter = 0

    # Get list of image files
    image_filenames = [f for f in os.listdir(train_image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]

    if not image_filenames:
        print(f"No image files found in {train_image_dir}. Skipping fold {fold_name} train data conversion.")
        continue

    for image_filename in image_filenames:
        image_path = os.path.join(train_image_dir, image_filename)

        # Use OpenCV to read image and get dimensions
        try:
            img = cv2.imread(image_path)
            if img is None:
                print(f"Warning: Could not read image {image_path}. Skipping.")
                continue
            img_height, img_width, _ = img.shape
        except Exception as e:
            print(f"Error reading image {image_path}: {e}. Skipping.")
            continue

        # Add image entry to COCO output
        coco_output['images'].append({
            'id': image_id_counter,
            'width': img_width,
            'height': img_height,
            'file_name': image_filename
        })

        # Construct path to corresponding YOLO label file
        label_filename = os.path.splitext(image_filename)[0] + '.txt'
        label_path = os.path.join(train_label_dir, label_filename)

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        x_center, y_center, width, height = map(float, parts[1:])

                        # Convert YOLO to COCO bbox
                        x_min, y_min, coco_width, coco_height = yolo_to_coco(
                            x_center, y_center, width, height, img_width, img_height
                        )

                        area = coco_width * coco_height

                        # Add annotation entry to COCO output
                        coco_output['annotations'].append({
                            'id': annotation_id_counter,
                            'image_id': image_id_counter,
                            'category_id': class_id,
                            'bbox': [x_min, y_min, coco_width, coco_height],
                            'area': area,
                            'iscrowd': 0
                        })
                        annotation_id_counter += 1
        else:
            print(f"Warning: Label file not found for {image_filename} at {label_path}. No annotations added for this image.")

        image_id_counter += 1

    # Save COCO JSON file for train data
    coco_train_output_path = os.path.join(fold_dir, 'COCO_train.json')
    with open(coco_train_output_path, 'w') as f:
        json.dump(coco_output, f, indent=4)
    print(f"Successfully created {coco_train_output_path} with {len(coco_output['images'])} images and {len(coco_output['annotations'])} annotations.")



Converting train data for fold_0 to COCO format...
Successfully created /content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_0/train/COCO_train.json with 492 images and 91871 annotations.

Converting train data for fold_1 to COCO format...


 **Convert Valid Data to COCO**



In [ ]:
for fold_name, fold_data in all_folds_data.items():
    print(f"\nConverting valid data for {fold_name} to COCO format...")

    valid_image_dir = fold_data['valid_image_dir']
    valid_label_dir = fold_data['valid_label_dir']
    categories = fold_data['categories']

    # Determine the fold_dir where COCO_valid.json should be saved
    # This is slightly different from train as the output is in the fold_dir, not valid/images
    fold_dir_path = os.path.dirname(os.path.dirname(valid_image_dir)) # Go up two levels from 'valid/images'

    coco_output = {
        'images': [],
        'annotations': [],
        'categories': categories
    }

    image_id_counter = 0
    annotation_id_counter = 0

    # Get list of image files
    image_filenames = [f for f in os.listdir(valid_image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]

    if not image_filenames:
        print(f"No image files found in {valid_image_dir}. Skipping fold {fold_name} valid data conversion.")
        continue

    for image_filename in image_filenames:
        image_path = os.path.join(valid_image_dir, image_filename)

        # Use OpenCV to read image and get dimensions
        try:
            img = cv2.imread(image_path)
            if img is None:
                print(f"Warning: Could not read image {image_path}. Skipping.")
                continue
            img_height, img_width, _ = img.shape
        except Exception as e:
            print(f"Error reading image {image_path}: {e}. Skipping.")
            continue

        # Add image entry to COCO output
        coco_output['images'].append({
            'id': image_id_counter,
            'width': img_width,
            'height': img_height,
            'file_name': image_filename
        })

        # Construct path to corresponding YOLO label file
        label_filename = os.path.splitext(image_filename)[0] + '.txt'
        label_path = os.path.join(valid_label_dir, label_filename)

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        x_center, y_center, width, height = map(float, parts[1:])

                        # Convert YOLO to COCO bbox
                        x_min, y_min, coco_width, coco_height = yolo_to_coco(
                            x_center, y_center, width, height, img_width, img_height
                        )

                        area = coco_width * coco_height

                        # Add annotation entry to COCO output
                        coco_output['annotations'].append({
                            'id': annotation_id_counter,
                            'image_id': image_id_counter,
                            'category_id': class_id,
                            'bbox': [x_min, y_min, coco_width, coco_height],
                            'area': area,
                            'iscrowd': 0
                        })
                        annotation_id_counter += 1
        else:
            print(f"Warning: Label file not found for {image_filename} at {label_path}. No annotations added for this image.")

        image_id_counter += 1

    # Save COCO JSON file for valid data
    coco_valid_output_path = os.path.join(fold_dir_path, 'COCO_valid.json')
    with open(coco_valid_output_path, 'w') as f:
        json.dump(coco_output, f, indent=4)
    print(f"Successfully created {coco_valid_output_path} with {len(coco_output['images'])} images and {len(coco_output['annotations'])} annotations.")

# Subset: Full Dataset

In [2]:
all_folds_data = {}

for i in range(5):
    fold_name = f'fold_{i}'
    fold_dir = os.path.join('/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data', fold_name)
    print(f"\nProcessing {fold_name}...")

    # Ensure fold_dir exists
    os.makedirs(fold_dir, exist_ok=True)

    # Define base paths for images and labels
    train_image_dir = os.path.join(fold_dir, 'train', 'images')
    train_label_dir = os.path.join(fold_dir, 'train', 'labels')
    valid_image_dir = os.path.join(fold_dir, 'valid', 'images')
    valid_label_dir = os.path.join(fold_dir, 'valid', 'labels')

    # Construct path to data.yaml
    data_yaml_path = os.path.join(fold_dir, 'data.yaml')

    # Create a dummy data.yaml if it doesn't exist to prevent FileNotFoundError
    if not os.path.exists(data_yaml_path):
        dummy_data = {
            'names': ['class1', 'class2', 'class3'],
            'nc': 3
        }
        with open(data_yaml_path, 'w') as f:
            yaml.safe_dump(dummy_data, f)
        print(f"Created dummy data.yaml at {data_yaml_path}.")

    # Load class names
    class_names = load_class_names(data_yaml_path)
    print(f"Loaded {len(class_names)} class names from {data_yaml_path}.")

    # Create COCO categories list
    categories = []
    for idx, name in enumerate(class_names):
        categories.append({
            'id': idx,
            'name': name,
            'supercategory': 'none'
        })
    print(f"Generated {len(categories)} COCO categories.")

    # Store paths and categories for the current fold
    all_folds_data[fold_name] = {
        'train_image_dir': train_image_dir,
        'train_label_dir': train_label_dir,
        'valid_image_dir': valid_image_dir,
        'valid_label_dir': valid_label_dir,
        'categories': categories,
        'class_names': class_names # Storing for potential debugging/future use
    }


Processing fold_0...
Loaded 31 class names from /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_0/data.yaml.
Generated 31 COCO categories.

Processing fold_1...
Loaded 31 class names from /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_1/data.yaml.
Generated 31 COCO categories.

Processing fold_2...
Loaded 31 class names from /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_2/data.yaml.
Generated 31 COCO categories.

Processing fold_3...
Loaded 31 class names from /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_3/data.yaml.
Generated 31 COCO categories.

Processing fold_4...
Loaded 31 class names from /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_4/data.yaml.
Generated 31 COCO categories.


** Convert Train Data to COCO**




In [3]:
for fold_name, fold_data in all_folds_data.items():
    print(f"\nConverting train data for {fold_name} to COCO format...")

    train_image_dir = fold_data['train_image_dir']
    train_label_dir = fold_data['train_label_dir']
    categories = fold_data['categories']
    fold_dir = os.path.dirname(train_image_dir.replace('/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_0/train/images', '/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_0/train/labels'))

    coco_output = {
        'images': [],
        'annotations': [],
        'categories': categories
    }

    image_id_counter = 0
    annotation_id_counter = 0

    # Get list of image files
    image_filenames = [f for f in os.listdir(train_image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]

    if not image_filenames:
        print(f"No image files found in {train_image_dir}. Skipping fold {fold_name} train data conversion.")
        continue

    for image_filename in image_filenames:
        image_path = os.path.join(train_image_dir, image_filename)

        # Use OpenCV to read image and get dimensions
        try:
            img = cv2.imread(image_path)
            if img is None:
                print(f"Warning: Could not read image {image_path}. Skipping.")
                continue
            img_height, img_width, _ = img.shape
        except Exception as e:
            print(f"Error reading image {image_path}: {e}. Skipping.")
            continue

        # Add image entry to COCO output
        coco_output['images'].append({
            'id': image_id_counter,
            'width': img_width,
            'height': img_height,
            'file_name': image_filename
        })

        # Construct path to corresponding YOLO label file
        label_filename = os.path.splitext(image_filename)[0] + '.txt'
        label_path = os.path.join(train_label_dir, label_filename)

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        x_center, y_center, width, height = map(float, parts[1:])

                        # Convert YOLO to COCO bbox
                        x_min, y_min, coco_width, coco_height = yolo_to_coco(
                            x_center, y_center, width, height, img_width, img_height
                        )

                        area = coco_width * coco_height

                        # Add annotation entry to COCO output
                        coco_output['annotations'].append({
                            'id': annotation_id_counter,
                            'image_id': image_id_counter,
                            'category_id': class_id,
                            'bbox': [x_min, y_min, coco_width, coco_height],
                            'area': area,
                            'iscrowd': 0
                        })
                        annotation_id_counter += 1
        else:
            print(f"Warning: Label file not found for {image_filename} at {label_path}. No annotations added for this image.")

        image_id_counter += 1

    # Save COCO JSON file for train data
    coco_train_output_path = os.path.join(fold_dir, 'COCO_train.json')
    with open(coco_train_output_path, 'w') as f:
        json.dump(coco_output, f, indent=4)
    print(f"Successfully created {coco_train_output_path} with {len(coco_output['images'])} images and {len(coco_output['annotations'])} annotations.")



Converting train data for fold_0 to COCO format...
Successfully created /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_0/train/COCO_train.json with 492 images and 99770 annotations.

Converting train data for fold_1 to COCO format...
Successfully created /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_1/train/COCO_train.json with 492 images and 104165 annotations.

Converting train data for fold_2 to COCO format...
Successfully created /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_2/train/COCO_train.json with 492 images and 102389 annotations.

Converting train data for fold_3 to COCO format...
Successfully created /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_3/train/COCO_train.json with 492 images and 101058 annotations.

Converting train data for fold_4 to COCO format...
Successfully created /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_4/train/COCO_train.json with 492 images and 98706 annotation

 **Convert Valid Data to COCO**



In [ ]:
for fold_name, fold_data in all_folds_data.items():
    print(f"\nConverting valid data for {fold_name} to COCO format...")

    valid_image_dir = fold_data['valid_image_dir']
    valid_label_dir = fold_data['valid_label_dir']
    categories = fold_data['categories']

    # Determine the fold_dir where COCO_valid.json should be saved
    # This is slightly different from train as the output is in the fold_dir, not valid/images
    fold_dir_path = os.path.dirname(os.path.dirname(valid_image_dir)) # Go up two levels from 'valid/images'

    coco_output = {
        'images': [],
        'annotations': [],
        'categories': categories
    }

    image_id_counter = 0
    annotation_id_counter = 0

    # Get list of image files
    image_filenames = [f for f in os.listdir(valid_image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]

    if not image_filenames:
        print(f"No image files found in {valid_image_dir}. Skipping fold {fold_name} valid data conversion.")
        continue

    for image_filename in image_filenames:
        image_path = os.path.join(valid_image_dir, image_filename)

        # Use OpenCV to read image and get dimensions
        try:
            img = cv2.imread(image_path)
            if img is None:
                print(f"Warning: Could not read image {image_path}. Skipping.")
                continue
            img_height, img_width, _ = img.shape
        except Exception as e:
            print(f"Error reading image {image_path}: {e}. Skipping.")
            continue

        # Add image entry to COCO output
        coco_output['images'].append({
            'id': image_id_counter,
            'width': img_width,
            'height': img_height,
            'file_name': image_filename
        })

        # Construct path to corresponding YOLO label file
        label_filename = os.path.splitext(image_filename)[0] + '.txt'
        label_path = os.path.join(valid_label_dir, label_filename)

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        x_center, y_center, width, height = map(float, parts[1:])

                        # Convert YOLO to COCO bbox
                        x_min, y_min, coco_width, coco_height = yolo_to_coco(
                            x_center, y_center, width, height, img_width, img_height
                        )

                        area = coco_width * coco_height

                        # Add annotation entry to COCO output
                        coco_output['annotations'].append({
                            'id': annotation_id_counter,
                            'image_id': image_id_counter,
                            'category_id': class_id,
                            'bbox': [x_min, y_min, coco_width, coco_height],
                            'area': area,
                            'iscrowd': 0
                        })
                        annotation_id_counter += 1
        else:
            print(f"Warning: Label file not found for {image_filename} at {label_path}. No annotations added for this image.")

        image_id_counter += 1

    # Save COCO JSON file for valid data
    coco_valid_output_path = os.path.join(fold_dir_path, 'COCO_valid.json')
    with open(coco_valid_output_path, 'w') as f:
        json.dump(coco_output, f, indent=4)
    print(f"Successfully created {coco_valid_output_path} with {len(coco_output['images'])} images and {len(coco_output['annotations'])} annotations.")


Converting valid data for fold_0 to COCO format...
Successfully created /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_0/COCO_valid.json with 123 images and 26752 annotations.

Converting valid data for fold_1 to COCO format...
Successfully created /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_1/COCO_valid.json with 123 images and 22357 annotations.

Converting valid data for fold_2 to COCO format...


# Subset: Missing Only

In [ ]:
all_folds_data = {}

for i in range(5):
    fold_name = f'fold_{i}'
    fold_dir = os.path.join('/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data', fold_name)
    print(f"\nProcessing {fold_name}...")

    # Ensure fold_dir exists
    os.makedirs(fold_dir, exist_ok=True)

    # Define base paths for images and labels
    train_image_dir = os.path.join(fold_dir, 'train', 'images')
    train_label_dir = os.path.join(fold_dir, 'train', 'labels')
    valid_image_dir = os.path.join(fold_dir, 'valid', 'images')
    valid_label_dir = os.path.join(fold_dir, 'valid', 'labels')

    # Construct path to data.yaml
    data_yaml_path = os.path.join(fold_dir, 'data.yaml')

    # Create a dummy data.yaml if it doesn't exist to prevent FileNotFoundError
    if not os.path.exists(data_yaml_path):
        dummy_data = {
            'names': ['class1', 'class2', 'class3'],
            'nc': 3
        }
        with open(data_yaml_path, 'w') as f:
            yaml.safe_dump(dummy_data, f)
        print(f"Created dummy data.yaml at {data_yaml_path}.")

    # Load class names
    class_names = load_class_names(data_yaml_path)
    print(f"Loaded {len(class_names)} class names from {data_yaml_path}.")

    # Create COCO categories list
    categories = []
    for idx, name in enumerate(class_names):
        categories.append({
            'id': idx,
            'name': name,
            'supercategory': 'none'
        })
    print(f"Generated {len(categories)} COCO categories.")

    # Store paths and categories for the current fold
    all_folds_data[fold_name] = {
        'train_image_dir': train_image_dir,
        'train_label_dir': train_label_dir,
        'valid_image_dir': valid_image_dir,
        'valid_label_dir': valid_label_dir,
        'categories': categories,
        'class_names': class_names # Storing for potential debugging/future use
    }

** Convert Train Data to COCO**




In [ ]:
for fold_name, fold_data in all_folds_data.items():
    print(f"\nConverting train data for {fold_name} to COCO format...")

    train_image_dir = fold_data['train_image_dir']
    train_label_dir = fold_data['train_label_dir']
    categories = fold_data['categories']
    fold_dir = os.path.dirname(train_image_dir.replace('/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/train/images', '/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/train/labels'))

    coco_output = {
        'images': [],
        'annotations': [],
        'categories': categories
    }

    image_id_counter = 0
    annotation_id_counter = 0

    # Get list of image files
    image_filenames = [f for f in os.listdir(train_image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]

    if not image_filenames:
        print(f"No image files found in {train_image_dir}. Skipping fold {fold_name} train data conversion.")
        continue

    for image_filename in image_filenames:
        image_path = os.path.join(train_image_dir, image_filename)

        # Use OpenCV to read image and get dimensions
        try:
            img = cv2.imread(image_path)
            if img is None:
                print(f"Warning: Could not read image {image_path}. Skipping.")
                continue
            img_height, img_width, _ = img.shape
        except Exception as e:
            print(f"Error reading image {image_path}: {e}. Skipping.")
            continue

        # Add image entry to COCO output
        coco_output['images'].append({
            'id': image_id_counter,
            'width': img_width,
            'height': img_height,
            'file_name': image_filename
        })

        # Construct path to corresponding YOLO label file
        label_filename = os.path.splitext(image_filename)[0] + '.txt'
        label_path = os.path.join(train_label_dir, label_filename)

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        x_center, y_center, width, height = map(float, parts[1:])

                        # Convert YOLO to COCO bbox
                        x_min, y_min, coco_width, coco_height = yolo_to_coco(
                            x_center, y_center, width, height, img_width, img_height
                        )

                        area = coco_width * coco_height

                        # Add annotation entry to COCO output
                        coco_output['annotations'].append({
                            'id': annotation_id_counter,
                            'image_id': image_id_counter,
                            'category_id': class_id,
                            'bbox': [x_min, y_min, coco_width, coco_height],
                            'area': area,
                            'iscrowd': 0
                        })
                        annotation_id_counter += 1
        else:
            print(f"Warning: Label file not found for {image_filename} at {label_path}. No annotations added for this image.")

        image_id_counter += 1

    # Save COCO JSON file for train data
    coco_train_output_path = os.path.join(fold_dir, 'COCO_train.json')
    with open(coco_train_output_path, 'w') as f:
        json.dump(coco_output, f, indent=4)
    print(f"Successfully created {coco_train_output_path} with {len(coco_output['images'])} images and {len(coco_output['annotations'])} annotations.")


 **Convert Valid Data to COCO**



In [ ]:
for fold_name, fold_data in all_folds_data.items():
    print(f"\nConverting valid data for {fold_name} to COCO format...")

    valid_image_dir = fold_data['valid_image_dir']
    valid_label_dir = fold_data['valid_label_dir']
    categories = fold_data['categories']

    # Determine the fold_dir where COCO_valid.json should be saved
    # This is slightly different from train as the output is in the fold_dir, not valid/images
    fold_dir_path = os.path.dirname(os.path.dirname(valid_image_dir)) # Go up two levels from 'valid/images'

    coco_output = {
        'images': [],
        'annotations': [],
        'categories': categories
    }

    image_id_counter = 0
    annotation_id_counter = 0

    # Get list of image files
    image_filenames = [f for f in os.listdir(valid_image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]

    if not image_filenames:
        print(f"No image files found in {valid_image_dir}. Skipping fold {fold_name} valid data conversion.")
        continue

    for image_filename in image_filenames:
        image_path = os.path.join(valid_image_dir, image_filename)

        # Use OpenCV to read image and get dimensions
        try:
            img = cv2.imread(image_path)
            if img is None:
                print(f"Warning: Could not read image {image_path}. Skipping.")
                continue
            img_height, img_width, _ = img.shape
        except Exception as e:
            print(f"Error reading image {image_path}: {e}. Skipping.")
            continue

        # Add image entry to COCO output
        coco_output['images'].append({
            'id': image_id_counter,
            'width': img_width,
            'height': img_height,
            'file_name': image_filename
        })

        # Construct path to corresponding YOLO label file
        label_filename = os.path.splitext(image_filename)[0] + '.txt'
        label_path = os.path.join(valid_label_dir, label_filename)

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        x_center, y_center, width, height = map(float, parts[1:])

                        # Convert YOLO to COCO bbox
                        x_min, y_min, coco_width, coco_height = yolo_to_coco(
                            x_center, y_center, width, height, img_width, img_height
                        )

                        area = coco_width * coco_height

                        # Add annotation entry to COCO output
                        coco_output['annotations'].append({
                            'id': annotation_id_counter,
                            'image_id': image_id_counter,
                            'category_id': class_id,
                            'bbox': [x_min, y_min, coco_width, coco_height],
                            'area': area,
                            'iscrowd': 0
                        })
                        annotation_id_counter += 1
        else:
            print(f"Warning: Label file not found for {image_filename} at {label_path}. No annotations added for this image.")

        image_id_counter += 1

    # Save COCO JSON file for valid data
    coco_valid_output_path = os.path.join(fold_dir_path, 'COCO_valid.json')
    with open(coco_valid_output_path, 'w') as f:
        json.dump(coco_output, f, indent=4)
    print(f"Successfully created {coco_valid_output_path} with {len(coco_output['images'])} images and {len(coco_output['annotations'])} annotations.")

# Subset: Non Missing

In [ ]:
all_folds_data = {}

for i in range(5):
    fold_name = f'fold_{i}'
    fold_dir = os.path.join('/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data', fold_name)
    print(f"\nProcessing {fold_name}...")

    # Ensure fold_dir exists
    os.makedirs(fold_dir, exist_ok=True)

    # Define base paths for images and labels
    train_image_dir = os.path.join(fold_dir, 'train', 'images')
    train_label_dir = os.path.join(fold_dir, 'train', 'labels')
    valid_image_dir = os.path.join(fold_dir, 'valid', 'images')
    valid_label_dir = os.path.join(fold_dir, 'valid', 'labels')

    # Construct path to data.yaml
    data_yaml_path = os.path.join(fold_dir, 'data.yaml')

    # Create a dummy data.yaml if it doesn't exist to prevent FileNotFoundError
    if not os.path.exists(data_yaml_path):
        dummy_data = {
            'names': ['class1', 'class2', 'class3'],
            'nc': 3
        }
        with open(data_yaml_path, 'w') as f:
            yaml.safe_dump(dummy_data, f)
        print(f"Created dummy data.yaml at {data_yaml_path}.")

    # Load class names
    class_names = load_class_names(data_yaml_path)
    print(f"Loaded {len(class_names)} class names from {data_yaml_path}.")

    # Create COCO categories list
    categories = []
    for idx, name in enumerate(class_names):
        categories.append({
            'id': idx,
            'name': name,
            'supercategory': 'none'
        })
    print(f"Generated {len(categories)} COCO categories.")

    # Store paths and categories for the current fold
    all_folds_data[fold_name] = {
        'train_image_dir': train_image_dir,
        'train_label_dir': train_label_dir,
        'valid_image_dir': valid_image_dir,
        'valid_label_dir': valid_label_dir,
        'categories': categories,
        'class_names': class_names # Storing for potential debugging/future use
    }

**Convert Train Data to COCO**




In [ ]:
for fold_name, fold_data in all_folds_data.items():
    print(f"\nConverting train data for {fold_name} to COCO format...")

    train_image_dir = fold_data['train_image_dir']
    train_label_dir = fold_data['train_label_dir']
    categories = fold_data['categories']
    fold_dir = os.path.dirname(train_image_dir.replace('/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_0/train/images', '/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_0/train/labels'))

    coco_output = {
        'images': [],
        'annotations': [],
        'categories': categories
    }

    image_id_counter = 0
    annotation_id_counter = 0

    # Get list of image files
    image_filenames = [f for f in os.listdir(train_image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]

    if not image_filenames:
        print(f"No image files found in {train_image_dir}. Skipping fold {fold_name} train data conversion.")
        continue

    for image_filename in image_filenames:
        image_path = os.path.join(train_image_dir, image_filename)

        # Use OpenCV to read image and get dimensions
        try:
            img = cv2.imread(image_path)
            if img is None:
                print(f"Warning: Could not read image {image_path}. Skipping.")
                continue
            img_height, img_width, _ = img.shape
        except Exception as e:
            print(f"Error reading image {image_path}: {e}. Skipping.")
            continue

        # Add image entry to COCO output
        coco_output['images'].append({
            'id': image_id_counter,
            'width': img_width,
            'height': img_height,
            'file_name': image_filename
        })

        # Construct path to corresponding YOLO label file
        label_filename = os.path.splitext(image_filename)[0] + '.txt'
        label_path = os.path.join(train_label_dir, label_filename)

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        x_center, y_center, width, height = map(float, parts[1:])

                        # Convert YOLO to COCO bbox
                        x_min, y_min, coco_width, coco_height = yolo_to_coco(
                            x_center, y_center, width, height, img_width, img_height
                        )

                        area = coco_width * coco_height

                        # Add annotation entry to COCO output
                        coco_output['annotations'].append({
                            'id': annotation_id_counter,
                            'image_id': image_id_counter,
                            'category_id': class_id,
                            'bbox': [x_min, y_min, coco_width, coco_height],
                            'area': area,
                            'iscrowd': 0
                        })
                        annotation_id_counter += 1
        else:
            print(f"Warning: Label file not found for {image_filename} at {label_path}. No annotations added for this image.")

        image_id_counter += 1

    # Save COCO JSON file for train data
    coco_train_output_path = os.path.join(fold_dir, 'COCO_train.json')
    with open(coco_train_output_path, 'w') as f:
        json.dump(coco_output, f, indent=4)
    print(f"Successfully created {coco_train_output_path} with {len(coco_output['images'])} images and {len(coco_output['annotations'])} annotations.")



Converting train data for fold_0 to COCO format...
Successfully created /content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_0/train/COCO_train.json with 257 images and 49413 annotations.

Converting train data for fold_1 to COCO format...


 **Convert Valid Data to COCO**



In [ ]:
for fold_name, fold_data in all_folds_data.items():
    print(f"\nConverting valid data for {fold_name} to COCO format...")

    valid_image_dir = fold_data['valid_image_dir']
    valid_label_dir = fold_data['valid_label_dir']
    categories = fold_data['categories']

    # Determine the fold_dir where COCO_valid.json should be saved
    # This is slightly different from train as the output is in the fold_dir, not valid/images
    fold_dir_path = os.path.dirname(os.path.dirname(valid_image_dir)) # Go up two levels from 'valid/images'

    coco_output = {
        'images': [],
        'annotations': [],
        'categories': categories
    }

    image_id_counter = 0
    annotation_id_counter = 0

    # Get list of image files
    image_filenames = [f for f in os.listdir(valid_image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]

    if not image_filenames:
        print(f"No image files found in {valid_image_dir}. Skipping fold {fold_name} valid data conversion.")
        continue

    for image_filename in image_filenames:
        image_path = os.path.join(valid_image_dir, image_filename)

        # Use OpenCV to read image and get dimensions
        try:
            img = cv2.imread(image_path)
            if img is None:
                print(f"Warning: Could not read image {image_path}. Skipping.")
                continue
            img_height, img_width, _ = img.shape
        except Exception as e:
            print(f"Error reading image {image_path}: {e}. Skipping.")
            continue

        # Add image entry to COCO output
        coco_output['images'].append({
            'id': image_id_counter,
            'width': img_width,
            'height': img_height,
            'file_name': image_filename
        })

        # Construct path to corresponding YOLO label file
        label_filename = os.path.splitext(image_filename)[0] + '.txt'
        label_path = os.path.join(valid_label_dir, label_filename)

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        x_center, y_center, width, height = map(float, parts[1:])

                        # Convert YOLO to COCO bbox
                        x_min, y_min, coco_width, coco_height = yolo_to_coco(
                            x_center, y_center, width, height, img_width, img_height
                        )

                        area = coco_width * coco_height

                        # Add annotation entry to COCO output
                        coco_output['annotations'].append({
                            'id': annotation_id_counter,
                            'image_id': image_id_counter,
                            'category_id': class_id,
                            'bbox': [x_min, y_min, coco_width, coco_height],
                            'area': area,
                            'iscrowd': 0
                        })
                        annotation_id_counter += 1
        else:
            print(f"Warning: Label file not found for {image_filename} at {label_path}. No annotations added for this image.")

        image_id_counter += 1

    # Save COCO JSON file for valid data
    coco_valid_output_path = os.path.join(fold_dir_path, 'COCO_valid.json')
    with open(coco_valid_output_path, 'w') as f:
        json.dump(coco_output, f, indent=4)
    print(f"Successfully created {coco_valid_output_path} with {len(coco_output['images'])} images and {len(coco_output['annotations'])} annotations.")